In [ ]:
pip install transformers scikit-learn torch numpy pandas peft ollama

In [1]:
#CALL MODEL

#choose parent model
model_name = "llama3.1:8b-instruct-q8_0"

#pull mode
!ollama pull {model_name}

!ollama list

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling cc04e85e1f86: 100% ▕██████████████████▏ 8.5 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 4a4a958ae550: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 
NAME                                                  ID              SIZE      MODIFIED               
llama3.1:8b-instruct-q8_0                             b158ded76fa0    8.5 GB    Less than a second ago    
llama3.1:8b                                           46e0c10c039e    4.9 GB    2 weeks ago               
llama3.1:8b-instruct-q8_0_framing_ft_frame6           39e5b7c0dc1d    8.5 GB    2 weeks ago               
llama3.1:8

In [ ]:
#LOAD DATA AND PACKAGES

#import packages and datasets
import pandas as pd
import numpy as np
import sklearn as sk
import json
import ollama
from pathlib import Path
from typing import Sequence
import re

#Unlabelled dataset
df_test = pd.read_csv("unlabelled_frames_test.csv")
#Labelled dataset
df_train = pd.read_csv('labelled_frames_train.csv')
df_test_lab = pd.read_csv('labelled_frames_test.csv')
df_gold = pd.concat([df_train, df_test_lab], axis=0).reset_index(drop=True)
#Frame names
frame_names = df_gold.columns[7:14].to_list()
#Codebook
with open('framing_codebook_short.json', 'r') as f:
    codebook = json.load(f)
codebook_str = json.dumps(codebook, indent=2, ensure_ascii=False)

In [ ]:
#HELPER FUNCTIONS

##parsing responses from model
def parse_json_with_fallback(content_str, frame_names):
    #Strip whitespace
    content_str = content_str.strip()
    # If the string is supposed to end with '}', but doesn't, add it.
    if not content_str.endswith('}'):
        content_str += '}'

    # Now try parsing
    try:
        #if the response can be parsed
        dict_val = json.loads(content_str)
        #remove any leading or trailing white spaces in the keys
        cleaned_dict = {k.strip(): v for k, v in dict_val.items()}

        annotations = []

        #parse response for each frame
        for f in frame_names:
            try:
                f_val = str(cleaned_dict.get(f))

                binary_val = pd.NA
                if re.search("yes",f_val.strip().lower()):
                    binary_val = 1
                if re.search("no",f_val.strip().lower()):
                    binary_val = 0

            except:
                binary_val = pd.NA
            
            annotations.append(binary_val)

        return annotations
    
    except json.JSONDecodeError:
        return [pd.NA]*len(frame_names)



##compute kappa and accuracy
def compute_scores(df_pred, df_gold, column_pred, column_gold, column_match):
    df_pred_reduced = df_pred[[column_match, column_pred]]
    df_gold_reduced = df_gold[[column_match, column_gold]]

    #join these two dataframes
    df_temp = (
        df_pred_reduced
            .merge(df_gold_reduced, on=column_match, how='inner', suffixes = ('_pred', '_gold'))   # keep only matching IDs
            .dropna(subset=[column_pred + '_pred', column_gold + '_pred'])          # drop rows where values are NaN
            .reset_index(drop=True)                                 # tidy up the index
    )
    
    #ensure that the columns have data in the same type 
    df_temp[column_pred + '_pred'] = df_temp[column_pred + '_pred'].astype(int)
    df_temp[column_gold + '_gold'] = df_temp[column_gold + '_gold'].astype(int)

    #compute scores
    acc = round(100*sk.metrics.accuracy_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    k = round(sk.metrics.cohen_kappa_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    f1 = round(sk.metrics.f1_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    precision = round(sk.metrics.precision_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    recall = round(sk.metrics.recall_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)

    print("Accuracy for " + column_gold +  " is " + str(acc) + "%, and Cohen's kappa is " + str(k))

    return([acc, k, f1, precision, recall])

#formatting fine tuning dataset to JSONL and saving
def to_ollama_jsonl(df_in: pd.DataFrame,
                    out_path: str | Path,
                    system_prompt: str,
                    user_prompt: str,
                    frame_names: Sequence[str],) -> Path:
    """
    Convert <article text, label> rows into Ollama‑ready JSONL.
    Each line is one conversation:  system ➜ user ➜ assistant
    """
    out_path = Path(out_path)
    with out_path.open("w", encoding="utf‑8") as f:
        for _, row in df_in.iterrows():
            answer = {}
            for frame in frame_names:
                label = row[frame]

                answer[frame] = "yes" if label == 1 else "no"

            record = {
                "messages": [
                    {"role": "system",    "content": system_prompt},
                    {"role": "user",      "content": user_prompt + row["text"]},
                    {"role": "assistant", "content": answer},
                ]
            }
            json.dump(record, f, ensure_ascii=False)
            f.write("\n")
    return out_path

In [ ]:
#LLM AS A JUDGE: HAVE TWO LLM CLASSIFIERS ANNOTATE THE ARTICLES AND THEN HAVE ANOTHER LLM JUDGE THE RESPONSES

SYSTEM_PROMPT = """You're a communication researcher who is studying the news reporting of Mpox. You’ll perform a codebook assisted framing analysis on news articles. 
Identify the frames from the codebook that are present in the article. Here is the codebook with frame definitions: """ + codebook_str


USER_PROMPT =  """Identify frames in the article with the following guidelines:
1. Read the entire article carefully before coding
2. Identify all frames present in each article
3. Some articles maybe irrelevant to the Mpox epidemic or may not contain any of the frames mentioned in the codebook. In this case, mark "no" for every frame. 
4. Ensure at least 2 framing dimensions are explicitly present (unless noted otherwise)
5. Use frame descriptions and examples to guide decisions
6. All coding should be based on EXPLICIT presence of the frame. Thus, if a frame isn't explicitly present, but implicitly implied, than mark "no" for that frame. 

Provide your response in a JSON array format with the answers and justifications, as follows, and include nothing else in the response: 
[
     {
          "frame": "sexual stigma and transmission routes",
          "presence": "yes/no",
          "justification": "a detailed chain of thought explaining your decision",
          "quote": "if the frame is present, an example quote from the article demonstrating it"
     },

     {
          "frame": "racial disparities and stigmatising name",
          "presence": "yes/no",
          "justification": "a detailed chain of thought explaining your decision",
          "quote": "if the frame is present, an example quote from the article demonstrating it"
     },

     {
          "frame": "global relations",
          "presence": "yes/no",
          "justification": "a detailed chain of thought explaining your decision",
          "quote": "if the frame is present, an example quote from the article demonstrating it"
     },

     {
          "frame": "public health failure",
          "presence": "yes/no",
          "justification": "a detailed chain of thought explaining your decision",
          "quote": "if the frame is present, an example quote from the article demonstrating it"
     },

     {
          "frame": "epidemic preparedness and surveillance",
          "presence": "yes/no",
          "justification": "a detailed chain of thought explaining your decision",
          "quote": "if the frame is present, an example quote from the article demonstrating it"
     },

     {
          "frame": "human-interest stories",
          "presence": "yes/no",
          "justification": "a detailed chain of thought explaining your decision",
          "quote": "if the frame is present, an example quote from the article demonstrating it"
     },

     {
          "frame": "broader health issues",
          "presence": "yes/no",
          "justification": "a detailed chain of thought explaining your decision",
          "quote": "if the frame is present, an example quote from the article demonstrating it"
     }
].

If you are uncertain about the annotations for any frame, force a decision to choose "yes" or "no" and then report that uncertainty in your justification. """


#sample for testing things first
#df_sample = df_test.sample(n= 30, random_state= 42).reset_index(drop=True)

#otherwise
df_sample = df_test.copy()
df1 = df_sample

predictions = list()

for i in range(len(df1)):
    try:
        #CLASSIFIERS
        classifier_out = list()
        for c_id in range(2):
             messages = [
                  {
                       "role": "system",
                       "content": SYSTEM_PROMPT
                  },

                  {
                       "role": "user",
                       "content":  USER_PROMPT + df1["text"][i]
                  }
             ]
             outputs = ollama.chat(model= model_name, messages= messages)
             classifier_out.append(outputs.message.content)


        #JUDGE
        messages = [
             {
                  "role": "system",
                  "content": """You are a communication researcher who is studying news framing surrounding the Mpox epidemic. 
                    Two classifiers have read a newspaper article and and identified whether certain frames from a codebook are present in the article or not. 
                    Your job is to go over their responses, chains of thought and the codebook yourself and make the final decision regarding whether the frames are present in the article. 
                    The classifiers tend to overlook the "explicit" clause in the codebook that mentions that any presence of frames must be based on explicit presence, so ensure that you check for that. 
                    Remember to prioritize accuracy and clarity in your analysis, using the provided context and your expertise to guide your evaluation.

                    Here is the original codebook used for analysis: """ + codebook_str
             },

             {
                  "role": "user",
                  "content": """ Here are the responses from the two classifiers: 
                    Classifier 1: """ + classifier_out[0] + """
                    Classifier 2: """ + classifier_out[1] + """

                    Provide your response in a JSON array format, as follows, and include nothing else in the response: 
                    {"sexual stigma and transmission routes": "yes/no",
                    "racial disparities and stigmatising name": "yes/no",
                    "global relations": "yes/no",
                    "public health failure": "yes/no",
                    "epidemic preparedness and surveillance": "yes/no",
                    "human-interest stories": "yes/no",
                    "broader health issues": "yes/no"}.

                    If you are uncertain about the annotations for any frame, force a decision to choose "yes" or "no". """
             }
        ]

        outputs = ollama.chat(model= model_name, messages= messages)
        predictions.append(outputs.message.content)
            

    #if there is an error
    except Exception as e:
            predictions.append(None)

    
    if(i%10 == 0): print(str(i) + " iterations finished")


#save the responses
df1[frame_names] = pd.NA

for j in range(len(predictions)):
    content_str = predictions[j].lower()
    annotation = parse_json_with_fallback(content_str, frame_names)
    df1.loc[j, frame_names] = annotation
    
#what are the scores looking like
scores_df = pd.DataFrame(index = frame_names,
                         columns= ['Accuracy', 'Kappa', 'F1'])

for f in frame_names:
    scores_df.loc[f] = compute_scores(df_gold = df_gold, df_pred = df1, column_match= "stories_id", column_gold= f, column_pred= f)
    
#save files
df1.to_csv("Predicted/Test/" + model_name + "_LLM_judge.csv")


In [ ]:
#HUMAN AS JUDGE: HAVE LLM ANNOTATE ARTICLES IN ROUND 1 WITH CHAIN OF THOUGHT (COT) AND THEN HAVE HUMAN EXPERT RESPOND TO THE ANNOTATIONS AND COT, AND LLM ANNOTATES AGAIN IN ROUND 2 

#ROUND 1
SYSTEM_PROMPT = """You're a communication researcher who is studying the news reporting of Mpox. You’ll perform a codebook assisted framing analysis on news articles. 
Identify the frames from the codebook that are present in the article. Here is the codebook with frame definitions: """ + codebook_str

#prompt with COT justification
USER_PROMPT =  """Identify frames in the article with the following guidelines:
1. Read the entire article carefully before coding
2. Identify all frames present in each article
3. Some articles maybe irrelevant to the Mpox epidemic or may not contain any of the frames mentioned in the codebook. In this case, mark "no" for every frame. 
4. Ensure at least 2 framing dimensions are explicitly present (unless noted otherwise)
5. Use frame descriptions and examples to guide decisions
6. All coding should be based on EXPLICIT presence of the frame. Thus, if a frame isn't explicitly present, but implicitly implied, than mark "no" for that frame. 

Provide your response in a JSON array format with the answers and justifications, as follows, and include nothing else in the response: 
[
     {
          "frame": "sexual stigma and transmission routes",
          "presence": "yes/no",
          "justification": "a detailed chain of thought explaining your decision",
          "quote": "if the frame is present, an example quote from the article demonstrating it"
     },

     {
          "frame": "racial disparities and stigmatizing name",
          "presence": "yes/no",
          "justification": "a detailed chain of thought explaining your decision",
          "quote": "if the frame is present, an example quote from the article demonstrating it"
     },

     {
          "frame": "global relations",
          "presence": "yes/no",
          "justification": "a detailed chain of thought explaining your decision",
          "quote": "if the frame is present, an example quote from the article demonstrating it"
     },

     {
          "frame": "public health failure",
          "presence": "yes/no",
          "justification": "a detailed chain of thought explaining your decision",
          "quote": "if the frame is present, an example quote from the article demonstrating it"
     },

     {
          "frame": "epidemic preparedness and surveillance",
          "presence": "yes/no",
          "justification": "a detailed chain of thought explaining your decision",
          "quote": "if the frame is present, an example quote from the article demonstrating it"
     },

     {
          "frame": "human-interest stories",
          "presence": "yes/no",
          "justification": "a detailed chain of thought explaining your decision",
          "quote": "if the frame is present, an example quote from the article demonstrating it"
     },

     {
          "frame": "broader health issues",
          "presence": "yes/no",
          "justification": "a detailed chain of thought explaining your decision",
          "quote": "if the frame is present, an example quote from the article demonstrating it"
     }
].

If you are uncertain about the annotations for any frame, force a decision to choose "yes" or "no" and then report that uncertainty in your justification. """

#sample a few articles for round 1 of annotation
df_sample = df_train.sample(n= 30, random_state= 42).reset_index(drop=True)
#df_sample.to_csv("Error/Human-judge/Round1_human_responses.csv") #save sample (predictions here are still human)

df1 = df_sample

predictions = list()

for i in range(len(df1)):
    try:
        messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": USER_PROMPT + df1["text"][i]
            }
        ]
        outputs = ollama.chat(model=model_name, messages=messages)
        classifier_out = outputs.message.content

        predictions.append(classifier_out)

    except Exception as e:
        predictions.append(None)

    if (i % 1 == 0):
        print(str(i) + " iterations finished")

#save the response 
with open("Error/Human-judge/Round1_llm_response.json", "w") as f:
         json.dump(predictions, f)

#NOW MANUALLY ASSESS THE OUTCOMES: 
#Human responses saved in "Error/Human-judge/Round1_human_responses.csv" 

#ROUND 2
#RE-ANNOTATE ARTICLES AFTER FINE-TUNING MODEL WITH THE EXCHANGE BETWEEN LLM AND HUMAN IN ROUND 1

SYSTEM_PROMPT = """You're a communication researcher who is studying the news reporting of Mpox. You’ll perform a codebook assisted framing analysis on news articles. 
Identify the frames from the codebook that are present in the article. Here is the codebook with frame definitions: """ + codebook_str


USER_PROMPT =  """Identify frames in the article with the following guidelines:
1. Read the entire article carefully before coding
2. Identify all frames present in each article
3. Some articles maybe irrelevant to the Mpox epidemic or may not contain any of the frames mentioned in the codebook. In this case, mark "no" for every frame. 
4. Ensure at least 2 framing dimensions are explicitly present (unless noted otherwise)
5. Use frame descriptions and examples to guide decisions
6. All coding should be based on EXPLICIT presence of the frame. Thus, if a frame isn't explicitly present, but implicitly implied, than mark "no" for that frame. 

Provide your response in a JSON array format, as follows, and include nothing else in the response: 
{"sexual stigma and transmission routes": "yes/no",
"racial disparities and stigmatising name": "yes/no",
"global relations": "yes/no",
"public health failure": "yes/no",
"epidemic preparedness and surveillance": "yes/no",
"human-interest stories": "yes/no",
"broader health issues": "yes/no"}.

If you are uncertain about the annotations for any frame, force a decision to choose "yes" or "no". """


#formatting fine tuning dataset to JSONL and saving
#read human responses to the round 1 predictions
human_df = pd.read_csv('Error/Human-judge/Round1_human_response.csv')
#convert binaries to string
yes_nos = human_df[frame_names].replace({1: "yes", 0: "no"})

#read llm responses
with open('Error/Human-judge/Round1_llm_response.json', 'r') as f:
    predictions = json.load(f)

messages_list = list()
 
for i in range(len(human_df)):
    row_dict = yes_nos.iloc[i].to_dict()
    #correct annotations
    json_dict = json.dumps(row_dict, indent= 2)

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT

        },
        {
            "role": "user",
            "content": USER_PROMPT + human_df["text"][i]
        },
        {
            "role": "assistant",
            "content": predictions[i]
        },
        {
            "role": "user",
            "content": human_df.loc[i,'Response'] 
        },
        {
            "role": "assistant",
            "content": "I see" + ". Then, the correct annotations would be: " + json_dict
        }
    ]

    #add these messages 
    messages_list.extend(messages)

#create file for finetuning
messages_str = json.dumps(messages_list, indent = 4)
record = {
    "messages": messages_str
}

#save file 
tune_path = "Error/Human-judge/framing_finetune_human_agentic.jsonl"
with open(tune_path, "w") as f:
        json.dump(record, f)

#Let the fine tuning begin
ft_model_name = model_name + "_framing_ft_human_agentic"

client = ollama.Client()                       # defaults to http://localhost:11434
digest = client.create_blob(tune_path)         # uploads the JSONL, returns sha256
progress = client.create(
    model       = ft_model_name,     # new tag
    from_       = model_name,                  # parent model
    files       = {"framing_finetune_human_agentic.jsonl": digest},
    parameters  = {"num_epochs": 3},           # any ggml‑compatible training args
    stream      = True                         # chunked progress
)
for chunk in progress:                         # stream shows download / training bar
    print(chunk.status, chunk.completed, "/", chunk.total)

#NOW ANNOTATE TARGET TEXTS

#sample some for testing the code first
#df_sample = df_test.sample(n= 10, random_state= 42).reset_index(drop=True)
#otherwise
df_sample = df_test.copy()

df1 = df_sample

predictions = list()

for i in range(len(df1)):
    try:
        messages = [
        {"role": "system", 
         "content": SYSTEM_PROMPT
        },

        {
         "role": "user", 
         "content": USER_PROMPT + df1["text"][i] 
        }

            ]
    
        outputs = ollama.chat(model= ft_model_name, messages= messages)

        predictions.append(outputs.message.content)
    
    #if there is an error
    except Exception as e:
            predictions.append(None)

    
    if(i%10 == 0): print(str(i) + " iterations finished")


#save the responses
df1[frame_names] = pd.NA

for j in range(len(predictions)):
    content_str = predictions[j].lower()
    annotation = parse_json_with_fallback(content_str, frame_names)
    df1.loc[j, frame_names] = annotation
    
#what are the scores looking like
scores_df = pd.DataFrame(index = frame_names,
                         columns= ['Accuracy', 'Kappa', 'F1', 'Precision', 'Recall'])

for f in frame_names:
    scores_df.loc[f] = compute_scores(df_gold = df_gold, df_pred = df1, column_match= "stories_id", column_gold= f, column_pred= f)
    

#save files
df1.to_csv("Predicted/Test/" + model_name + "_human_judge.csv")


In [ ]:
#DECISION TREE WITH YES/NO QUESTIONS: ONE-TO-ONE ON FRAMES

#load the decision tree
with open('Error/framing_decision_tree.json', 'r') as f:
    decision_tree = json.load(f)

#convert to dataframe
df_tree = pd.DataFrame(decision_tree)

#Load data
#sample for testing things first
#df_sample = df_test.sample(n= 5, random_state= 42).reset_index(drop=True)

#otherwise
df_sample = df_test.copy()
df1 = df_sample

for f in frame_names:
    #obtain question and output format
    frame_tree = df_tree[df_tree['frame'] == f]

    #questions formatted
    frame_questions = json.dumps(dict(frame_tree['question']), indent=2)

    #output formatted
    frame_output = json.dumps(dict(zip(frame_tree['question'], ['yes/no'] * len(frame_tree))), indent=2)

    SYSTEM_PROMPT = """You're a communication researcher who is studying the news reporting of Mpox. You’ll be analysing an article and answering a series of "yes or no" questions. 
    Here are the questions: """ + frame_questions

    USER_PROMPT = """Ensure that you follow these guidelines:
    1. Read the entire article carefully before answering
    2. Some articles maybe irrelevant to the Mpox epidemic. In this case, mark "no" for every question.
    3. Answer each and every question that is asked.
    4. All answers must be based on an EXPLICIT interpretation of the article. Thus, if the answer for a question is "yes", but only based on what is implicitly implied, mark "no" for that question.
    5. If you are uncertain about the answer for any question, force a decision to choose "yes" or "no".
    
    Provide your response in a JSON array format, as follows, and include nothing else in the response: """ + frame_output + """ 
    Here's the article: """

    predictions = list()

    for i in range(len(df1)):
        try:
            messages = [
            {"role": "system", 
            "content": SYSTEM_PROMPT
            },

            {
            "role": "user", 
            "content": USER_PROMPT + df1["text"][i] 
            }

                ]
        
            outputs = ollama.chat(model= model_name, messages= messages)

            predictions.append(outputs.message.content)
        
        #if there is an error
        except Exception as e:
                predictions.append(None)
        
        #stream progress
        if(i%10 == 0): print(str(i) + " iterations finished")


    #add responses to the dataframe
    df1[f] = pd.NA

    for j in range(len(predictions)):
        article_tree = frame_tree
        content_str = predictions[j].lower()

        #obtain the answers for that article
        article_tree.loc[:,'answer'] = parse_json_with_fallback(content_str, frame_tree.loc[:,'question'])

        #if any answer in a dimension is 1, then that dimension is covered
        result = article_tree.groupby('dimension')['answer'].max()

        #if any of the dimensions are NA, then return annotation for that frame is NA
        if any(result == pd.NA):
            annotation = pd.NA
        
        else:
            percentage_ones = result.mean()
            annotation = 1 if percentage_ones >= 0.5 else 0

        df1.loc[j, f] = annotation

    
    print(f + ' annotations finished')
    
    
#what are the scores looking like
scores_df = pd.DataFrame(index = frame_names,
                         columns= ['Accuracy', 'Kappa', 'F1'])

for f in frame_names:
    scores_df.loc[f] = compute_scores(df_gold = df_gold, df_pred = df1, column_match= "stories_id", column_gold= f, column_pred= f)
    
#save files
df1.to_csv("Predicted/Test/" + model_name + "_decision_tree.csv")


In [ ]:
#ZERO SHOT SCORES: ASK MODEL TO OUTPUT PROBABILITIES OF PREVALENCE FOR EACH FRAME AND THEN LEARN THE BEST THRESHOLD USING A TRAIN SET


#TRAIN

#change parsing function for confidence scores as opposed to binary "yes/no"
def parse_json_with_fallback_scores(content_str, frame_names):
    #Strip whitespace
    content_str = content_str.strip()
    # If the string is supposed to end with '}', but doesn't, add it.
    if not content_str.endswith('}'):
        content_str += '}'

    # Now try parsing
    try:
        #if the response can be parsed
        dict_val = json.loads(content_str)
        #remove any leading or trailing white spaces in the keys
        cleaned_dict = {k.strip(): v for k, v in dict_val.items()}

        annotations = []

        #parse response for each frame
        for f in frame_names:
            try:
                f_val = cleaned_dict.get(f)

            except:
                f_val = pd.NA
            
            annotations.append(f_val)

        return annotations
    
    except json.JSONDecodeError:
        return [pd.NA]*len(frame_names)

#obtain a sample of articles from the train set to first get confidence scores for and determine the threshold
df_sample = df_train.sample(n= 100, random_state= 42).reset_index(drop=True)

df1 = df_sample[df_test.columns.to_list()] #only keep columns that can be in the test set

for frame in frame_names:
    codebook_str_f = json.dumps(codebook[frame_names.index(frame)], indent=2, ensure_ascii=False)

    SYSTEM_PROMPT = """You're a communication researcher who is studying the news reporting of Mpox. You’ll perform a codebook assisted framing analysis on news articles. 
    You're interested in detecting the presence of the frame: """ + frame + """. Here is the codebook definition of the frame: """ + codebook_str_f

    USER_PROMPT = """Identify the probability that the frame (""" + frame + """) is present in the news article using the following guidelines:
     1. Read the entire article carefully before coding
     2. Some articles maybe irrelevant to the Mpox epidemic. In this case, mark '0'.  
     3. Ensure at least 2 framing dimensions are explicitly present (unless noted otherwise)
     4. Use frame description and examples to guide decisions

     Provide your response in a JSON array format as follows, and include nothing else in the response: 
     {"confidence": "A number between 0 (definitely absent) and 1 (definitely present) indicating the probability that the frame exists in the article"}

     Here's the article: """

    predictions = list()

    for i in range(len(df1)):
        try:
            messages = [
            {"role": "system", 
            "content": SYSTEM_PROMPT
            },

            {
            "role": "user", 
            "content": USER_PROMPT + df1["text"][i] 
            }

                ]
        
            outputs = ollama.chat(model= model_name, messages= messages)

            predictions.append(outputs.message.content)
        
        #if there is an error
        except Exception as e:
                predictions.append(None)
        
        #stream progress
        if(i%10 == 0): print(str(i) + " iterations finished")


    #save the responses
    df1[frame] = pd.NA

    for j in range(len(predictions)):
        content_str = predictions[j].lower()
        annotation = parse_json_with_fallback_scores(content_str, ['confidence'])
        df1.loc[j, frame] = annotation[0]
    
    print(frame + ' annotations finished')
    

df1.to_csv("Error/Confidence/" + model_name + "_1to1_zshot_train.csv")

#DETERMINE THRESHOLDS NOW
thresh = np.round(np.linspace(0, 1, num=21), 2).tolist()
df_l_train = df1
df_thresh = pd.DataFrame(index=frame_names,
                         columns=['threshold', 'max_kappa']) #save the thresholds

for f in frame_names:
    metric_vals = list() #store list of values for the metric we're optimizing for
    df_f_train = df_l_train[df_test.columns.to_list() + [f]] #create temporary dataframe for just that frame
    for t in thresh:
        df_temp = df_f_train.copy()
        df_temp[f] = [1 if elem >= t else 0 for elem in df_f_train[f]]

        metric_vals.append(compute_scores(df_pred=df_temp, df_gold=df_train, column_pred=f, column_gold=f, column_match='stories_id')[1]) #computer score returns [accuracy, kappa, f1, precision, recall]
    
    df_thresh.loc[f, 'threshold'] = thresh[np.argmax(metric_vals)] #returns the first max value threshold
    df_thresh.loc[f, 'max_metric'] = max(metric_vals) #returns the first max value threshold


#NOW APPLY ON THE TEST SET
df1 = df_test.copy()

for frame in frame_names:
    codebook_str_f = json.dumps(codebook[frame_names.index(frame)], indent=2, ensure_ascii=False)
    
    SYSTEM_PROMPT = """You're a communication researcher who is studying the news reporting of Mpox. You’ll perform a codebook assisted framing analysis on news articles. 
    You're interested in detecting the presence of the frame: """ + frame + """. Here is the codebook definition of the frame: """ + codebook_str_f

    USER_PROMPT = """Identify the probability that the frame (""" + frame + """) is present in the news article using the following guidelines:
     1. Read the entire article carefully before coding
     2. Some articles maybe irrelevant to the Mpox epidemic. In this case, mark '0'.  
     3. Ensure at least 2 framing dimensions are explicitly present (unless noted otherwise)
     4. Use frame description and examples to guide decisions

     Provide your response in a JSON array format as follows, and include nothing else in the response: 
     {"confidence": "A number between 0 (definitely absent) and 1 (definitely present) indicating the probability that the frame exists in the article"}

     Here's the article: """

    predictions = list()

    for i in range(len(df1)):
        try:
            messages = [
            {"role": "system", 
            "content": SYSTEM_PROMPT
            },

            {
            "role": "user", 
            "content": USER_PROMPT + df1["text"][i] 
            }

                ]
        
            outputs = ollama.chat(model= model_name, messages= messages)

            predictions.append(outputs.message.content)
        
        #if there is an error
        except Exception as e:
                predictions.append(None)
        
        #stream progress
        if(i%10 == 0): print(str(i) + " iterations finished")


    #save the responses
    df1[frame] = pd.NA

    for j in range(len(predictions)):
        content_str = predictions[j].lower()
        annotation = parse_json_with_fallback_scores(content_str, ['confidence'])
        df1.loc[j, frame] = annotation[0]
    
    print(frame + ' annotations finished')

df1.to_csv("Error/Confidence/" + model_name + "_1to1_zshot_test.csv")    

#CHANGE CONFIDENCE SCORES TO BINARY VALUES BASED ON INDIVIDUAL THRESHOLDS
df2 = df1.copy()
for f in frame_names:
    df2.loc[:,f] = [1 if elem >= df_thresh.loc[f, 'threshold'] else 0 for elem in df1[f]]
    compute_scores(df_pred=df2, df_gold=df_gold, column_pred=f, column_gold=f, column_match='stories_id')

df2.to_csv("Predicted/Test/" + model_name + "_confidence.csv", index=False)


In [1]:
import json
with open('Error/framing_decision_tree.json', 'r') as f:
    decision_tree = json.load(f)